# Approval Processing
Explicitly validate a reviewed workbook, publish approved context versions, and archive the processed batch.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import load_app_config
from dq_agent.context_store import import_approval_workbook, read_context
from dq_agent.context_utils import (
    configure_workflow_logging, context_workflow_paths, logged_step, read_approval_workbook,
)

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
APPROVAL_FILE = None  # Example: ROOT / 'approvals/pending/context_approvals_....xlsx'
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)
pending_files = sorted(paths['pending'].glob('*.xlsx'))
display([str(path) for path in pending_files])

In [ ]:
if APPROVAL_FILE is None:
    raise ValueError('Set APPROVAL_FILE to one reviewed workbook listed above.')
APPROVAL_FILE = Path(APPROVAL_FILE)
with logged_step(logger, paths['checkpoint'], 'VALIDATE_APPROVAL_FILE', file=str(APPROVAL_FILE)):
    decisions = read_approval_workbook(APPROVAL_FILE)
display(decisions)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'IMPORT_APPROVAL_RESULTS', file=str(APPROVAL_FILE)):
    import_result = import_approval_workbook(config, APPROVAL_FILE, paths, logger)
display(import_result['summary'])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'PUBLISH_APPROVED_CONTEXT'):
    trusted = read_context(config, logger=logger)
    print(f'Current trusted records: {len(trusted)}')
display(trusted[['record_id','context_type','subject_key','version','origin','approved_by','approved_at']])

## Manual verification
Confirm the workbook moved to `approvals/processed`, receipts exist under `approved` or `rejected`, and rerunning the import does not create another context version.